# DLinear for v2 — package API walkthrough

This notebook shows how to train and forecast with **DLinear** using the v2
`TimeSeries` + `TslibDataModule` stack via `DLinear_pkg_v2`.

The v2 API is experimental and may change; see the [v2 tutorials index](../tutorials_v2.rst).

## Imports

In [ ]:
from sklearn.preprocessing import StandardScaler

from pytorch_forecasting.data.encoders import TorchNormalizer
from pytorch_forecasting.data.examples import load_toydata
from pytorch_forecasting.data.timeseries import TimeSeries
from pytorch_forecasting.metrics import MAE, SMAPE
from pytorch_forecasting.models.dlinear._dlinear_pkg_v2 import DLinear_pkg_v2

## Load toy data

DLinear currently focuses on continuous features, so we keep only numeric
columns from `load_toydata`.

In [ ]:
num_series = 100
seq_length = 50
data_df = load_toydata(num_series, seq_length)
data_df = data_df[
    [
        "series_id",
        "time_idx",
        "x",
        "y",
        "future_known_feature",
        "static_feature",
    ]
]
data_df.head()

## Create a `TimeSeries` dataset

`TimeSeries` is the D1 layer: it wraps the raw DataFrame and declares roles for
time, target, groups, and feature columns.

In [ ]:
dataset = TimeSeries(
    data=data_df,
    time="time_idx",
    target="y",
    group=["series_id"],
    num=["x", "future_known_feature", "static_feature"],
    cat=[],
    known=["future_known_feature"],
    unknown=["x"],
    static=["static_feature"],
)

## Configure datamodule, model, and trainer

`DLinear_pkg_v2` uses `TslibDataModule`, so datamodule kwargs are
`context_length` / `prediction_length` (not encoder/decoder lengths).
Include scalers and a target normalizer as in the other v2 tutorials.

In [ ]:
datamodule_cfg = dict(
    context_length=30,
    prediction_length=1,
    batch_size=32,
    add_relative_time_idx=True,
    scalers={
        "x": StandardScaler(),
        "future_known_feature": StandardScaler(),
        "static_feature": StandardScaler(),
    },
    target_normalizer=TorchNormalizer(),
)

model_cfg = dict(
    loss=MAE(),
    logging_metrics=[MAE(), SMAPE()],
    optimizer="adam",
    optimizer_params={"lr": 1e-3},
    moving_avg=25,
    individual=False,
)

trainer_cfg = dict(
    max_epochs=5,
    accelerator="auto",
    devices=1,
    enable_progress_bar=True,
    log_every_n_steps=10,
)

## Fit and predict with `DLinear_pkg_v2`

The package container wires the model, `TslibDataModule`, and Lightning trainer.
Pass the `TimeSeries` object to `fit` / `predict`.

In [ ]:
model_pkg = DLinear_pkg_v2(
    model_cfg=model_cfg,
    trainer_cfg=trainer_cfg,
    datamodule_cfg=datamodule_cfg,
)

model_pkg.fit(dataset)
preds = model_pkg.predict(dataset, return_info=["index", "x", "y"])
preds["prediction"][:3]